# iprPy elastic_constants_static calculation

In [1]:
# Standard library imports
import datetime

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-25 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('elastic_constants_static')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# elastic_constants_static calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The elastic_constants_static calculation style computes the elastic constants, $C_{ij}$, for a system by applying small strains and performing static energy minimizations of the initial and strained configurations.  Three estimates of the elastic constants are returned: one for applying positive strains, one for applying negative strains, and a normalized estimate that averages the &pm; strains and the symmetric components of the $C_{ij}$ tensor.

### Version notes

- 2018-07-09: Notebook added.
- 2019-07-30: Description updated and small changes due to iprPy version.
- v0.10.0: Version 0.10 update - potentials now loaded from database.
- 2020-09-22: Setup and parameter definition streamlined.
- v0.11.0: Notebook updated to reflect version 0.11.
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- Unlike the previous LAMMPS_ELASTIC calculation, this calculation does *not* perform a box relaxation on the system prior to evaluating the elastic constants.  This allows for the static elastic constants to be evaluated for systems that are relaxed to different pressures.
- The elastic constants are estimated using small strains.  Depending on the potential, the values for the elastic constants may vary with the size of the strain.  This can come about either if the strain exceeds the linear elastic regime.
- Some classical interatomic potentials have discontinuities in the fourth derivative of the energy function with respect to position.  If the strained states straddle one of these discontinuities the resulting static elastic constants values will be nonsense.


## Method and Theory

The calculation method used here for computing elastic constants is based on the method used in the ELASTIC demonstration script created by Steve Plimpton and distributed with LAMMPS.

The math in this section uses Voigt notation, where indicies i,j correspond to 1=xx, 2=yy, 3=zz, 4=yz, 5=xz, and 6=xy, and x, y and z are orthogonal box vectors.

A LAMMPS simulation performs thirteen energy/force minimizations

- One for relaxing the initial system.

- Twelve for relaxing systems in which a small strain of magnitude $\Delta \epsilon$ is applied to the system in both the positive and negative directions of the six Voigt strain components, $\pm \Delta \epsilon_{i}$.

The system virial pressures, $P_{i}$, are recorded for each of the thirteen relaxed configurations.  Two estimates for the $C_{ij}$ matrix for the system are obtained as

$$ C_{ij}^+ = - \frac{P_i(\Delta \epsilon_j) - P_i(0)}{\Delta \epsilon},$$

$$ C_{ij}^- = - \frac{P_i(0) - P_i(-\Delta \epsilon_j)}{\Delta \epsilon}.$$

The negative out front comes from the fact that the system-wide stress state is $\sigma_i = -P_i$.  A normalized, average estimate is also obtained by averaging the positive and negative strain estimates, as well as the symmetric components of the tensor

$$ C_{ij} = \frac{C_{ij}^+ + C_{ij}^- + C_{ji}^+ + C_{ji}^-}{4}.$$


## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "elastic_constants_static.py"

# Python script created by Lucas Hale
# Originally based on the LAMMPS example script by Steve Plimpton

# Standard library imports
from typing import Optional, Union

# http://www.numpy.org/
import numpy as np

# https://pandas.pydata.org/
import pandas as pd

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj

def elastic_constants_static(lammps_command: Union[str, LAMMPSobj],
                             system: am.System,
                             potential: lammpspotential,
                             mpi_command: Optional[str] = None,
                             strainrange: float = 1e-6,
                             etol: float = 0.0,
                             ftol: unitfloat = 0.0,
                             maxiter: int = 10000,
                             maxeval: int = 100000,
                             dmax: unitfloat = '0.01 angstrom',
                             usefiles: bool = True) -> dict:
    """
    Computes the elastic constants of an atomic configuration using small
    strains.  This calculation is comparable to the LAMMPS ELASTIC example.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The system to perform the calculation on.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    strainrange : float, optional
        The small strain value to apply when calculating the elastic
        constants (default is 1e-6).
    etol : float, optional
        The energy tolerance for the structure minimization. This value is
        unitless. (Default is 0.0).
    ftol : float, optional
        The force tolerance for the structure minimization. This value is in
        units of force. (Default is 0.0).
    maxiter : int, optional
        The maximum number of minimization iterations to use (default is 10000).
    maxeval : int, optional
        The maximum number of minimization evaluations to use (default is 
        100000).
    dmax : float, optional
        The maximum distance in length units that any atom is allowed to relax
        in any direction during a single minimization iteration (default is
        0.01 Angstroms).
    usefiles : bool, optional
        If set to True, then all input/output files for LAMMPS will be generated.
        Default value of False will minimize the files created.
    
    Returns
    -------
    dict
        Dictionary of results consisting of keys:
        
        - **'raw_Cij_negative'** (*numpy.ndarray*) - The values of Cij obtained
          from only the negative strains.
        - **'raw_Cij_positive'** (*numpy.ndarray*) - The values of Cij obtained
          from only the positive strains.
        - **'C'** (*atomman.ElasticConstants*) - The computed elastic constants
          obtained from averaging the negative and positive strain values.
    """
    # Create a LAMMPS object if needed
    lmp = LAMMPS(lammps_command, mpi_command=mpi_command, potential=potential)

    # Convert values given with units if needed
    ftol = uc.set_in_units(ftol)
    dmax = uc.set_in_units(dmax)

    # Convert hexagonal cells to orthorhombic to avoid LAMMPS tilt issues
    if am.tools.ishexagonal(system.box):
        system = system.rotate([[2,-1,-1,0], [0, 1, -1, 0], [0,0,0,1]])

    # Call exe or lib function version for LAMMPS calculation
    all_thermo = cij_static(lmp, system, potential,
                            strainrange=strainrange,
                            etol=etol, ftol=ftol, maxiter=maxiter,
                            maxeval=maxeval, dmax=dmax)
    

   

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'lmp_serial'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 3 Mar 2020


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial unit cell system

- __ucell__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is generated using the load parameters and symbols.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Create ucell by loading prototype record
ucell = am.load('crystal', potential=potential, family='A1--Cu--fcc')

print(ucell)

Multiple matching record retrieved from remote
#  family               symbols  alat    Ecoh    method  standing
 1 A1--Cu--fcc          Ni        3.5200 -4.4500 dynamic good
 2 A1--Cu--fcc          Ni        7.3760  0.0119 dynamic good


Please select one: 1


avect =  [ 3.520,  0.000,  0.000]
bvect =  [ 0.000,  3.520,  0.000]
cvect =  [ 0.000,  0.000,  3.520]
origin = [ 0.000,  0.000,  0.000]
natoms = 4
natypes = 1
symbols = ('Ni',)
pbc = [ True  True  True]
per-atom properties = ['atype', 'pos']
     id |   atype |  pos[0] |  pos[1] |  pos[2]
      0 |       1 |   0.000 |   0.000 |   0.000
      1 |       1 |   0.000 |   1.760 |   1.760
      2 |       1 |   1.760 |   0.000 |   1.760
      3 |       1 |   1.760 |   1.760 |   0.000


### 3.4. System modifications

- __sizemults__ list of three integers specifying how many times the ucell vectors of $a$, $b$ and $c$ are replicated in creating system.

- __system__ is an atomman.System to perform the scan on (required). 

In [9]:
sizemults = [3, 3, 3]

# Generate system by supersizing ucell
system = ucell.supersize(*sizemults)
print('# of atoms in system =', system.natoms)

# of atoms in system = 108


### 3.5. Calculation-specific parameters

- __strainrange__ specifies the $\Delta \epsilon$ strain range to use in estimating $C_{ij}$.
- __energytolerance__ is the energy tolerance to use during the minimizations. This is unitless.
- __forcetolerance__ is the force tolerance to use during the minimizations. This is in energy/length units.
- __maxiterations__ is the maximum number of minimization iterations to use.
- __maxevaluations__ is the maximum number of minimization evaluations to use.
- __maxatommotion__ is the largest distance that an atom is allowed to move during a minimization iteration. This is in length units.

In [10]:
strainrange = 1e-7
energytolerance = 1e-8
forcetolerance = uc.set_in_units(0.0, 'eV/angstrom')
maxiterations = 10000
maxevaluations = 100000
maxatommotion = uc.set_in_units(0.01, 'angstrom')

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [11]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.elastic_constants_static.elastic_constants_static'

In [12]:
results_dict = calculation.calc(lammps_command, system, potential,
                                mpi_command = mpi_command,
                                strainrange = strainrange,
                                etol = energytolerance,
                                ftol = forcetolerance,
                                maxiter = maxiterations,
                                maxeval = maxevaluations,
                                dmax = maxatommotion)
print(results_dict.keys())

dict_keys(['raw_Cij_negative', 'raw_Cij_positive', 'C'])


### 4.2. Report results

Values returned in the results_dict:

- **'raw_Cij_negative'** (*numpy.ndarray*) - The values of Cij obtained
  from only the negative strains.
- **'raw_Cij_positive'** (*numpy.ndarray*) - The values of Cij obtained
  from only the positive strains.
- **'C'** (*atomman.ElasticConstants*) - The computed elastic constants
  obtained from averaging the negative and positive strain values.

In [13]:
pressure_unit = 'GPa'

print('Raw Cij for negative strains ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['raw_Cij_negative'], pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))
print()

print('Raw Cij for positive strains ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['raw_Cij_positive'], pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))  

Raw Cij for negative strains (GPa) =
[ 247.8622  147.8285  147.8285   -0.0000   -0.0000   -0.0000]
[ 147.8285  247.8622  147.8285   -0.0000   -0.0000   -0.0000]
[ 147.8285  147.8285  247.8622   -0.0000   -0.0000   -0.0000]
[  -0.0000    0.0001    0.0001  124.8381   -0.0000   -0.0000]
[   0.0001   -0.0000    0.0001   -0.0000  124.8381   -0.0000]
[   0.0001    0.0001   -0.0000   -0.0000   -0.0000  124.8381]

Raw Cij for positive strains (GPa) =
[ 247.8625  147.8283  147.8283    0.0000    0.0000    0.0000]
[ 147.8283  247.8625  147.8283    0.0000    0.0000    0.0000]
[ 147.8283  147.8283  247.8625    0.0000    0.0000    0.0000]
[   0.0000   -0.0001   -0.0001  124.8381    0.0000    0.0000]
[  -0.0001    0.0000   -0.0001    0.0000  124.8381    0.0000]
[  -0.0001   -0.0001    0.0000    0.0000    0.0000  124.8381]


In [14]:
print('Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['C'].Cij, pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))

Cij (GPa) =
[ 247.8624  147.8284  147.8284    0.0000    0.0000    0.0000]
[ 147.8284  247.8624  147.8284    0.0000    0.0000    0.0000]
[ 147.8284  147.8284  247.8624    0.0000    0.0000    0.0000]
[   0.0000    0.0000    0.0000  124.8381    0.0000    0.0000]
[   0.0000    0.0000    0.0000    0.0000  124.8381    0.0000]
[   0.0000    0.0000    0.0000    0.0000    0.0000  124.8381]


In [15]:
family = am.tools.identifyfamily(ucell.box)
C = results_dict['C']

if not C.is_normal(family):
    print("Cij not consistent with ucell's box")

else:
    norm_C = C.normalized_as(family)
    
    if family == 'cubic':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
    elif family == 'hexagonal':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C13 =', uc.get_in_units(norm_C.Cij[0,2], 'GPa'), 'GPa')
        print('C33 =', uc.get_in_units(norm_C.Cij[2,2], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
    elif family == 'tetragonal':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C13 =', uc.get_in_units(norm_C.Cij[0,2], 'GPa'), 'GPa')
        print('C16 =', uc.get_in_units(norm_C.Cij[0,5], 'GPa'), 'GPa')
        print('C33 =', uc.get_in_units(norm_C.Cij[2,2], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
        print('C66 =', uc.get_in_units(norm_C.Cij[5,5], 'GPa'), 'GPa')
    elif family == 'rhombohedral':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C13 =', uc.get_in_units(norm_C.Cij[0,2], 'GPa'), 'GPa')
        print('C14 =', uc.get_in_units(norm_C.Cij[0,3], 'GPa'), 'GPa')
        print('C15 =', uc.get_in_units(norm_C.Cij[0,4], 'GPa'), 'GPa')
        print('C33 =', uc.get_in_units(norm_C.Cij[2,2], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
    elif family == 'orthorhombic':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C13 =', uc.get_in_units(norm_C.Cij[0,2], 'GPa'), 'GPa')
        print('C22 =', uc.get_in_units(norm_C.Cij[1,1], 'GPa'), 'GPa')
        print('C23 =', uc.get_in_units(norm_C.Cij[1,2], 'GPa'), 'GPa')
        print('C33 =', uc.get_in_units(norm_C.Cij[2,2], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
        print('C55 =', uc.get_in_units(norm_C.Cij[4,4], 'GPa'), 'GPa')
        print('C66 =', uc.get_in_units(norm_C.Cij[5,5], 'GPa'), 'GPa')
    elif family == 'monoclinic':
        print('C11 =', uc.get_in_units(norm_C.Cij[0,0], 'GPa'), 'GPa')
        print('C12 =', uc.get_in_units(norm_C.Cij[0,1], 'GPa'), 'GPa')
        print('C13 =', uc.get_in_units(norm_C.Cij[0,2], 'GPa'), 'GPa')
        print('C15 =', uc.get_in_units(norm_C.Cij[0,4], 'GPa'), 'GPa')
        print('C22 =', uc.get_in_units(norm_C.Cij[1,1], 'GPa'), 'GPa')
        print('C23 =', uc.get_in_units(norm_C.Cij[1,2], 'GPa'), 'GPa')
        print('C25 =', uc.get_in_units(norm_C.Cij[1,4], 'GPa'), 'GPa')
        print('C33 =', uc.get_in_units(norm_C.Cij[2,2], 'GPa'), 'GPa')
        print('C35 =', uc.get_in_units(norm_C.Cij[2,4], 'GPa'), 'GPa')
        print('C44 =', uc.get_in_units(norm_C.Cij[3,3], 'GPa'), 'GPa')
        print('C46 =', uc.get_in_units(norm_C.Cij[3,5], 'GPa'), 'GPa')
        print('C55 =', uc.get_in_units(norm_C.Cij[4,4], 'GPa'), 'GPa')
        print('C66 =', uc.get_in_units(norm_C.Cij[5,5], 'GPa'), 'GPa')
    else:
        print('system is triclinic: just look at Cij above')

C11 = 247.86236442082614 GPa
C12 = 147.82841322360173 GPa
C44 = 124.83811767733812 GPa


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [16]:
calculation.clean_files()